# Evaluate 2Wiki Predictions
Brief notebook to compute EM/F1 from saved predictions.

In [ ]:
from pathlib import Path
import json
import re
import string
from collections import Counter

In [ ]:
ROOT = Path.cwd()
PRED_PATH = ROOT / "results" / "2wiki_predictions_dev.jsonl"
SUMMARY_PATH = ROOT / "results" / "2wiki_eval_dev.json"

In [ ]:
def normalize_answer(text):
    text = (text or "").lower()
    text = "".join(ch for ch in text if ch not in set(string.punctuation))
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())

def exact_match(pred, gold):
    return float(normalize_answer(pred) == normalize_answer(gold))

def f1_score(pred, gold):
    pred_tokens = normalize_answer(pred).split()
    gold_tokens = normalize_answer(gold).split()
    if not pred_tokens and not gold_tokens:
        return 1.0
    if not pred_tokens or not gold_tokens:
        return 0.0
    common = Counter(pred_tokens) & Counter(gold_tokens)
    n_same = sum(common.values())
    if n_same == 0:
        return 0.0
    p = n_same / len(pred_tokens)
    r = n_same / len(gold_tokens)
    return 2 * p * r / (p + r)

In [ ]:
rows = []
with PRED_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

valid = [r for r in rows if not str(r.get("prediction", "")).startswith("__ERROR__:")]
errors = [r for r in rows if str(r.get("prediction", "")).startswith("__ERROR__:")]

em = sum(exact_match(r.get("prediction", ""), r.get("gold_answer", "")) for r in valid) / max(len(valid), 1)
f1 = sum(f1_score(r.get("prediction", ""), r.get("gold_answer", "")) for r in valid) / max(len(valid), 1)

summary = {
    "total": len(rows),
    "valid": len(valid),
    "errors": len(errors),
    "exact_match": em,
    "f1": f1,
}

SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
with SUMMARY_PATH.open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

summary